### import packages needed for the script

In [1]:
import pandas as pd
import csv
import os
import sys
import requests
from dotenv import load_dotenv
from datetime import date
import time

### Run this code only once, as it downloads and formats my mini project dataset

This code uses the Open Library Search API to collect book data related to artificial intelligence topics. It searches multiple AI-related keywords, such as artificial intelligence, machine learning, deep learning, neural networks, generative AI, and large language models. For each search topic, the code sends API requests to Open Library, retrieves book metadata, and extracts fields such as title, author name, first publication year, subjects, language, edition count, publisher, and publication place.

The code also limits the number of search pages and rows collected for each topic to keep the dataset manageable. It filters for recent books based on publication year, removes duplicate works using the Open Library work key, stores the results in a pandas DataFrame, and saves the final dataset as `miniproject_dataset.csv`.

In [2]:
# Open Library does not require an API key

# Settings
OUTPUT_CSV = "miniproject_dataset.csv"
MAX_ROWS_TOTAL = 1200
DEFAULT_ROWS_PER_TOPIC = 200
PER_PAGE = 100
TIMEOUT = 30
MAX_PAGES_PER_TOPIC = 8

# Give artificial intelligence more rows than other topics
TOPIC_ROW_LIMITS = {
    "artificial intelligence": 600,
}

# "Past year" approximation based on publication year.
# Open Library often gives year-level data, not exact publication dates.
CURRENT_YEAR = date.today().year
MIN_YEAR = CURRENT_YEAR - 1

# AI-related search topics
SEARCH_TOPICS = [
    "artificial intelligence",
    "machine learning",
    "deep learning",
    "neural networks",
    "generative AI",
    "large language models",
    "natural language processing",
    "NLP",
    "computer vision",
]

# Open Library Search API endpoint
url = "https://openlibrary.org/search.json"

# Fields useful for your analytical questions
fields = [
    "key",
    "title",
    "author_name",
    "first_publish_year",
    "publish_year",
    "subject",
    "language",
    "edition_count",
    "publisher",
    "place",
]

all_rows = []
seen_keys = set()

for topic in SEARCH_TOPICS:
    page = 1
    topic_rows_collected = 0
    topic_limit = TOPIC_ROW_LIMITS.get(topic, DEFAULT_ROWS_PER_TOPIC)

    while (
        len(all_rows) < MAX_ROWS_TOTAL
        and topic_rows_collected < topic_limit
        and page <= MAX_PAGES_PER_TOPIC
    ):
        params = {
            "q": topic,
            "fields": ",".join(fields),
            "sort": "new",
            "limit": PER_PAGE,
            "page": page,
        }

        try:
            response = requests.get(url, params=params, timeout=TIMEOUT)
        except requests.exceptions.RequestException as e:
            print(f"Connection error for topic '{topic}', page {page}: {e}")
            break

        if not response.ok:
            print(f"Request failed for topic '{topic}', page {page}")
            print(response.text)
            break

        data = response.json()
        docs = data.get("docs", [])

        if not docs:
            break

        for book in docs:
            if len(all_rows) >= MAX_ROWS_TOTAL:
                break

            if topic_rows_collected >= topic_limit:
                break

            work_key = book.get("key")

            # Skip duplicate works across different search topics
            if work_key in seen_keys:
                continue

            seen_keys.add(work_key)

            title = book.get("title")
            authors = book.get("author_name", [])
            first_publish_year = book.get("first_publish_year")
            publish_years = book.get("publish_year", [])
            subjects = book.get("subject", [])
            languages = book.get("language", [])
            publishers = book.get("publisher", [])
            places = book.get("place", [])
            edition_count = book.get("edition_count")

            # Recent book filter.
            # Open Library usually gives year-level data, not exact dates.
            is_recent = False

            if isinstance(first_publish_year, int) and first_publish_year >= MIN_YEAR:
                is_recent = True
            elif isinstance(publish_years, list):
                recent_years = [
                    y for y in publish_years
                    if isinstance(y, int) and y >= MIN_YEAR
                ]
                if recent_years:
                    is_recent = True

            if not is_recent:
                continue

            row = {
                "search_topic": topic,
                "openlibrary_work_key": work_key,
                "title": title,
                "author_name": "; ".join(authors) if isinstance(authors, list) else authors,
                "first_publish_year": first_publish_year,
                "publish_years": "; ".join(map(str, publish_years)) if isinstance(publish_years, list) else publish_years,
                "subjects": "; ".join(subjects[:25]) if isinstance(subjects, list) else subjects,
                "languages": "; ".join(languages) if isinstance(languages, list) else languages,
                "edition_count": edition_count,
                "publishers": "; ".join(publishers[:10]) if isinstance(publishers, list) else publishers,
                "publication_places": "; ".join(places[:10]) if isinstance(places, list) else places,
            }

            all_rows.append(row)
            topic_rows_collected += 1

        print(
            f"Topic: {topic} | Page: {page} | "
            f"Topic rows: {topic_rows_collected} | Total rows: {len(all_rows)}"
        )

        page += 1
        time.sleep(3)

        if len(docs) < PER_PAGE:
            break

    if page > MAX_PAGES_PER_TOPIC:
        print(f"Stopped '{topic}' after reaching {MAX_PAGES_PER_TOPIC} pages.")

# Convert to DataFrame
df = pd.DataFrame(all_rows)

# Extra cleanup
if not df.empty:
    df = df.drop_duplicates(subset=["openlibrary_work_key"])
    df = df.head(MAX_ROWS_TOTAL)

# Save CSV
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nSaved {len(df)} rows to {OUTPUT_CSV}")

print("\nRows by search topic:")
print(df["search_topic"].value_counts())

Topic: artificial intelligence | Page: 1 | Topic rows: 100 | Total rows: 100
Topic: artificial intelligence | Page: 2 | Topic rows: 200 | Total rows: 200
Topic: artificial intelligence | Page: 3 | Topic rows: 300 | Total rows: 300
Topic: artificial intelligence | Page: 4 | Topic rows: 400 | Total rows: 400
Topic: artificial intelligence | Page: 5 | Topic rows: 500 | Total rows: 500
Topic: artificial intelligence | Page: 6 | Topic rows: 600 | Total rows: 600
Topic: machine learning | Page: 1 | Topic rows: 75 | Total rows: 675
Topic: machine learning | Page: 2 | Topic rows: 161 | Total rows: 761
Topic: machine learning | Page: 3 | Topic rows: 174 | Total rows: 774
Topic: machine learning | Page: 4 | Topic rows: 178 | Total rows: 778
Topic: machine learning | Page: 5 | Topic rows: 181 | Total rows: 781
Topic: machine learning | Page: 6 | Topic rows: 185 | Total rows: 785
Topic: machine learning | Page: 7 | Topic rows: 187 | Total rows: 787
Topic: machine learning | Page: 8 | Topic rows: 1

### Get a brief summary of my dataset

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1134 entries, 0 to 1133
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   search_topic          1134 non-null   str  
 1   openlibrary_work_key  1134 non-null   str  
 2   title                 1134 non-null   str  
 3   author_name           1134 non-null   str  
 4   first_publish_year    1134 non-null   int64
 5   publish_years         1134 non-null   str  
 6   subjects              1134 non-null   str  
 7   languages             1134 non-null   str  
 8   edition_count         1134 non-null   int64
 9   publishers            1134 non-null   str  
 10  publication_places    1134 non-null   str  
dtypes: int64(2), str(9)
memory usage: 97.6 KB


There are 11 columns in total: 9 string-type columns and 2 integer-type columns. The `first_publish_year` column represents the year when the book was first published, and `edition_count` represents how many editions or versions Open Library has recorded for that work. All columns have 1,040 non-null rows, so there are no missing values detected in any column. However, this does not guarantee that the dataset has no placeholder values such as "NA" or "NULL", because these are stored as strings and would not be detected as null values automatically.

### Check the top 5 rows of the dataset.

In [4]:
df.head()

,search_topic,openlibrary_work_key,title,author_name,first_publish_year,publish_years,subjects,languages,edition_count,publishers,publication_places
0,artificial intelligence,/works/OL44867871W,The Mind Behind The Machine,Artificial Intelligence,2026,2026,Artificial Intelligence; Talal Abu-Ghazaleh; M...,eng,1,"Abu-Ghazaleh Translation, Distribution & Publi...",
1,artificial intelligence,/works/OL44711065W,Epistémologie de l'IA – New Entry SOTA First I...,Stefano Dorian Franco,2026,2026,Artificial intelligence; Epistemology; Philoso...,,1,Studio SFB creation multimedia / Amazon KDP,
2,artificial intelligence,/works/OL44977494W,How to Write a Paper,Andrew David Naselli,2026,2026,,eng,1,Build & Fight Press,
3,artificial intelligence,/works/OL44766535W,The Art Book,Lori Randolph,2026,2026,Modern Art; Graphic arts; Art collections; Art...,,1,Self-Published,
4,artificial intelligence,/works/OL45147574W,Transforming Medicinal Plant Agriculture,Pankaj Kumar; Ashish R. Warghat,2026,2026,Artificial intelligence; Social sciences; Engi...,eng,1,Springer,


Looks like some of the `publication_places` columns are empty.

In [5]:
df[df['publication_places']==''].head()

,search_topic,openlibrary_work_key,title,author_name,first_publish_year,publish_years,subjects,languages,edition_count,publishers,publication_places
0,artificial intelligence,/works/OL44867871W,The Mind Behind The Machine,Artificial Intelligence,2026,2026,Artificial Intelligence; Talal Abu-Ghazaleh; M...,eng,1,"Abu-Ghazaleh Translation, Distribution & Publi...",
1,artificial intelligence,/works/OL44711065W,Epistémologie de l'IA – New Entry SOTA First I...,Stefano Dorian Franco,2026,2026,Artificial intelligence; Epistemology; Philoso...,,1,Studio SFB creation multimedia / Amazon KDP,
2,artificial intelligence,/works/OL44977494W,How to Write a Paper,Andrew David Naselli,2026,2026,,eng,1,Build & Fight Press,
3,artificial intelligence,/works/OL44766535W,The Art Book,Lori Randolph,2026,2026,Modern Art; Graphic arts; Art collections; Art...,,1,Self-Published,
4,artificial intelligence,/works/OL45147574W,Transforming Medicinal Plant Agriculture,Pankaj Kumar; Ashish R. Warghat,2026,2026,Artificial intelligence; Social sciences; Engi...,eng,1,Springer,


Just as we expected, `publication_places` has rows with value '', empty string but the dataset can not detect it as null values.

In [6]:
import numpy as np
df_cleaned = df.copy()
df_cleaned["publication_places"] = df_cleaned["publication_places"].replace("", np.nan)

In [7]:
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 1134 entries, 0 to 1133
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   search_topic          1134 non-null   str  
 1   openlibrary_work_key  1134 non-null   str  
 2   title                 1134 non-null   str  
 3   author_name           1134 non-null   str  
 4   first_publish_year    1134 non-null   int64
 5   publish_years         1134 non-null   str  
 6   subjects              1134 non-null   str  
 7   languages             1134 non-null   str  
 8   edition_count         1134 non-null   int64
 9   publishers            1134 non-null   str  
 10  publication_places    15 non-null     str  
dtypes: int64(2), str(9)
memory usage: 97.6 KB


In [8]:
df_cleaned[~(df_cleaned['publication_places'].isna())]

,search_topic,openlibrary_work_key,title,author_name,first_publish_year,publish_years,subjects,languages,edition_count,publishers,publication_places
78,artificial intelligence,/works/OL45069465W,Don't Replace Me,Dmitry Kargaev,2026,2026,artificial intelligence; career development; A...,,1,Independently Published,Los Angeles
82,artificial intelligence,/works/OL45038630W,The Useless,,2026,2026,dystopia; dystopian fiction; science fiction; ...,,1,Joosep Wyrd,New Geneva; EU Compact; Vienna
95,artificial intelligence,/works/OL44913796W,Transition Age,Tyler Corriveau,2026,2026,Young adult science fiction; dystopian; post-a...,eng,1,Tyler Corriveau,Chicago (Ill.); Chicago Metropolitan Area; New...
100,artificial intelligence,/works/OL45267578W,"Predicting Injustice; Bias, Ethics, and the Ma...",Dr. Lincoln Eden,2026,2026,Predictive policing; Algorithmic bias; Artific...,eng,1,Cipher Press,United States; Los Angeles; Chicago; Plainfiel...
102,artificial intelligence,/works/OL45566003W,Algorithmic Invisibility,Drey Russell,2026,2026,Algorithm; Eccho Chambers; Bubble; SOCIAL SCIE...,,1,Always On Marketing & Creative Studio,New York (State)
108,artificial intelligence,/works/OL45142895W,Seven AI Laws The Future of Mankind,Adv Prashant Mali,2026,2026,"Artificial Intelligence; Legal status, laws; S...",eng,2,Cyber Infomedia,India; USA
194,artificial intelligence,/works/OL43676746W,Infinite War Epic,VAELIX,2025,2025,apocalyptic fiction; end of the world; Earth d...,,1,‎ Independently Published,Korea (South); Tokyo (Japan); Korea (North); C...
374,artificial intelligence,/works/OL44218882W,Immersion,,2025,2025,Artificial intelligence; dystopian; thriller; ...,,1,Amazon Direct Publishing,Washington (D.C.); Michigan Territory
375,artificial intelligence,/works/OL44229384W,Forging the Beast of Bay Area,,2025,2025,Business; Autobiography; Memoir; Entrepreneurs...,eng,1,Cent Capital Global Inc.,Bay Area; Silicon Valley; Seattle; New York Ci...
382,artificial intelligence,/works/OL44368253W,Künstliche Intelligenz und der neue Faschismus,Rainer Mühlhoff,2025,2025,Artificial intelligence; Fascism; Silicon Vall...,ger,1,Reclam-Verlag,Silicon Valley (California)


Since the majority of values in `publication_places` are missing, analyzing where the most recent AI-related books were published is not applicable for this dataset.

#### column `author_name` and `languages` also have alot of empty rows not detected by pandas

In [9]:
df_cleaned["author_name"] = df_cleaned["author_name"].replace("", np.nan)
df_cleaned["languages"] = df_cleaned["languages"].replace("", np.nan)
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 1134 entries, 0 to 1133
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   search_topic          1134 non-null   str  
 1   openlibrary_work_key  1134 non-null   str  
 2   title                 1134 non-null   str  
 3   author_name           1118 non-null   str  
 4   first_publish_year    1134 non-null   int64
 5   publish_years         1134 non-null   str  
 6   subjects              1134 non-null   str  
 7   languages             1080 non-null   str  
 8   edition_count         1134 non-null   int64
 9   publishers            1134 non-null   str  
 10  publication_places    15 non-null     str  
dtypes: int64(2), str(9)
memory usage: 97.6 KB


In [10]:
string_columns = df_cleaned.select_dtypes(include="str").columns
df_cleaned[string_columns] = df_cleaned[string_columns].replace("", np.nan)
df_cleaned.isna().sum()

search_topic               0
openlibrary_work_key       0
title                      0
author_name               16
first_publish_year         0
publish_years              0
subjects                 775
languages                 54
edition_count              0
publishers                 0
publication_places      1119
dtype: int64

fixed this empty string problem to all string type columns.

### what is the number of unique values in the edition count

In [11]:
df.edition_count.value_counts()

edition_count
1       775
2       187
3       105
4        30
5        27
6         3
8         2
7         2
1379      1
2420      1
18        1
Name: count, dtype: int64

For an AI-related topic, rows with `edition_count` values such as 1,378 or 2,420 are likely not reasonable. Artificial intelligence is a relatively recent field compared with classic literature, so it is unlikely that a truly AI-related book would have that many recorded editions or versions. These unusually high values may indicate noisy search results, duplicate catalog records, or books that were incorrectly matched to an AI-related search topic.

In [12]:
df[(df['edition_count']==1378) | (df['edition_count']==2420)]

,search_topic,openlibrary_work_key,title,author_name,first_publish_year,publish_years,subjects,languages,edition_count,publishers,publication_places
1064,large language models,/works/OL45089W,Robinson Crusoe,Daniel Defoe; J. J. Grandville; Petrus Borel; ...,1686,1686; 1719; 1720; 1722; 1724; 1735; 1736; 1737...,"Crusoe, robinson (fictitious character), ficti...",ger; heb; pol; spa; und; epo; rum; dut; gle; p...,2420,"printed for A. Millar, W. Law, and R. Cater; a...",Sweden; Foreign countries; Atlantic Ocean; Pac...


It appears that both `Through the Looking-Glass` and `Robinson Crusoe` were incorrectly included in the AI-related book dataset. This may have happened because the search topic `large language models` contains broad terms such as “language,” which can also match classic literature books.

### Remove the two books that were incorrectly included in the dataset

In [13]:
titles_to_remove = [
    "Through the Looking-Glass",
    "Robinson Crusoe",
]

# Remove rows where title matches those books
df_cleaned = df[~df["title"].isin(titles_to_remove)]

df_cleaned.to_csv("miniproject_dataset_cleaned.csv", index=False, encoding="utf-8")

In [14]:
df_cleaned.edition_count.value_counts()

edition_count
1     775
2     187
3     105
4      30
5      27
6       3
8       2
7       2
18      1
Name: count, dtype: int64

After cleaning, the dataset no longer includes these incorrectly matched books, such as `Through the Looking-Glass` and `Robinson Crusoe`.

### Double-check the 18 rows with high `edition_count` values to make sure they are not false search results.

In [15]:
df_cleaned[(df_cleaned['edition_count']==18)]

,search_topic,openlibrary_work_key,title,author_name,first_publish_year,publish_years,subjects,languages,edition_count,publishers,publication_places
1132,computer vision,/works/OL20810913W,Pattern Recognition and Computer Vision,Jian-Huang Lai; Cheng-Lin Liu; Xilin Chen; Jie...,2018,2018; 2019; 2020; 2021; 2022; 2024; 2026,Pattern recognition systems; Computer vision,eng,18,Springer International Publishing AG; Springer,


This is a correctly included AI related book.

### Analytical questions (cleaned MP1 dataset)

The cells below use **`df_cleaned`** after the cleaning steps above. Each question states what I want to know, runs pandas on that table, and summarizes the answer to match the printed output.


#### Question 1: Which AI-related search topic returned the most books after cleaning?

I want to see whether my Open Library pulls are balanced across topics or dominated by one query string.


In [16]:
topic_counts = df_cleaned["search_topic"].value_counts()
print(topic_counts)
print()
top_topic = topic_counts.index[0]
top_n = topic_counts.iloc[0]
print(f"Most rows: {top_topic!r} ({top_n} works, {100 * top_n / len(df_cleaned):.1f}% of the cleaned set)")


search_topic
artificial intelligence        600
machine learning               187
generative AI                  159
deep learning                   64
large language models           29
computer vision                 28
neural networks                 24
natural language processing     23
NLP                             18
Name: count, dtype: int64

Most rows: 'artificial intelligence' (600 works, 53.0% of the cleaned set)


**Answer:** **`artificial intelligence`** has the most rows (**600 of 1,132**, **53.0%**). Next are **machine learning** (187) and **generative AI** (159); the remaining topics are smaller. The pull is **not balanced across search strings**—the first topic got more API pages—so MP1 charts should not treat each topic as an equal sample size.


#### Question 2: How recent are the `first_publish_year` values in the cleaned set?

My API filter targeted newer material; I want to confirm the table still looks "recent" after removing bad matches.


In [17]:
print(df_cleaned["first_publish_year"].describe())
recent = df_cleaned[df_cleaned["first_publish_year"] >= 2020]
print(
    f"\nWorks with first_publish_year >= 2020: "
    f"{len(recent)} of {len(df_cleaned)} ({100 * len(recent) / len(df_cleaned):.1f}%)"
)


count    1132.000000
mean     2025.026502
std         0.754125
min      2015.000000
25%      2025.000000
50%      2025.000000
75%      2025.000000
max      2026.000000
Name: first_publish_year, dtype: float64

Works with first_publish_year >= 2020: 1128 of 1132 (99.6%)


**Answer:** **`first_publish_year`** ranges **2015–2026**; the median and mean are both **2025**. **1,128 of 1,132 rows (99.6%)** are **2020 or later**—only **4 rows** are older than 2020. That fits the recent-year filter from collection; this is not a long historical sample of AI books.


#### Question 3: What languages are recorded, and how often is `languages` blank or missing?

Language matters for accessibility and for knowing whether English-only search terms biased what we pulled. I count both **NaN** and **empty strings** because pandas treats them differently.


In [18]:
lang_counts = df_cleaned["languages"].value_counts(dropna=False)
print(lang_counts.head(10))

missing_lang = df_cleaned["languages"].isna().sum()
blank_lang = (df_cleaned["languages"].fillna("").astype(str).str.strip() == "").sum()
print(f"\nRows with NaN language: {missing_lang}")
print(f"Rows with blank language (''): {blank_lang} ({100 * blank_lang / len(df_cleaned):.1f}%)")


languages
eng                   1059
                        54
ger                      6
ita                      4
tur                      2
fre                      2
spa; eng                 1
spa                      1
fre; lat; ita; eng       1
ita; eng                 1
Name: count, dtype: int64

Rows with missing language: 0 (0.0%)


**Answer:** **`eng`** is most common (**1,059 rows**). A few multi-language codes appear (for example **`spa; eng`**). **`isna()` reports 0 missing**, but **`value_counts` shows 54 blank rows**—empty strings pandas still counts as a category, not null (same issue as earlier in cleaning). Non-English single codes (**ger**, **ita**, etc.) are a small tail. I would not treat this as a language-balanced sample for MP1 without noting the **~4.8%** blank `languages` rows.



#### Summary

In this notebook I **downloaded** Open Library AI-topic data, **cleaned** it with pandas (empty strings, false title matches, edition_count checks), **saved** `miniproject_dataset_cleaned.csv`, and **answered three analytical questions** on **`df_cleaned`** (1,132 rows in my run).
